In [ ]:
from datasets import Dataset

In [ ]:
data=[
    {
        "instruction": "Match resume with job",
        "input": "Skills: Python, ML | Job: Data Scientist with SQL",
        "output": "Match: 75%, Missing Skills: SQL"
    },
    {
        "instruction": "Match resume with job",
        "input": "Skills: Java, Spring | Job: Backend Developer with Node.js",
        "output": "Match: 60%, Missing Skills: Node.js"
    },
    {
        "instruction": "Match resume with job",
        "input": "Skills: HTML, CSS | Job: Frontend Developer with React",
        "output": "Match: 65%, Missing Skills: React"
    }
]
dataset=Dataset.from_list(data)

In [ ]:
print(dataset)

!pip install -U bitsandbytes accelerate transformers peft

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM,BitsAndBytesConfig
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer=AutoTokenizer.from_pretrained(model_name)


# Creating the  4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Loading the model with config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
def format_example(example):
    return {
        "text": f"""
Instruction: {example['instruction']}
Input: {example['input']}
Output: {example['output']}
"""
    }

dataset = dataset.map(format_example)

In [ ]:
#applying the lora  technique of fine tuning
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [ ]:
input_text = """
Instruction: Match resume with job
Input: Skills: Python, ML | Job: AI Engineer with Deep Learning
Output:
"""

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=20)

print(tokenizer.decode(output[0]))